# 5.11 同一段文字出現很多次會怎樣？


三筆原文是「貓  看狗」「貓 看狗」「狗 看貓」。前兩筆只差空白數量，卻可能被當新內容、切到訓練與驗證兩側。我們先明說可忽略的差異，再數不同內容。

本例只把連續空白合成一個空格，這叫文字正規化；不改字序和字義，也與整理向量尺度的數值正規化不同。

程式先用 `split()` 按空白分段，再用 `" ".join(...)` 以單一空格接回。為了快速比較內容，`fingerprint` 也先做同樣的整理，再用SHA-256算出固定長的內容指紋，也叫hash。因此，可以把原文直接交給它：前兩段經過整理後相同，應得到同一個指紋，最後的assert就是核對這件事。它不是把下方的 `normalized` 清單拿來當輸入，而是在函式內整理每份原文。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import fingerprint

docs = ["貓  看狗", "貓 看狗", "狗 看貓"]
normalized = [" ".join(text.split()) for text in docs]
hashes = [fingerprint(text) for text in docs]
print(normalized)
print("原始筆數", len(docs), "正規化後不同內容", len(set(hashes)))
print("指紋字元數", len(hashes[0]))
assert hashes[0] == hashes[1]

第一行應顯示兩份 `貓 看狗` 與一份 `狗 看貓`。三筆記錄因此只有兩種不同內容，前兩個完整指紋相同。每個完整指紋有64個十六進位字元，也就是每格使用0–9或a–f這16種符號之一。

我們常用指紋比較長文件，不必每次逐字掃描。它不是可還原的文字壓縮，也不證明兩段文字在意思上相同；換一個字往往會讓指紋整體改變。碰撞指兩份不同的正規化內容偶然得到同一個完整指紋；SHA-256發生這種情況的機會極小，但判定重要重複時仍可保留原文核對。只取極短前綴則丟掉了許多區分資訊，更容易讓不同內容對上相同短字串，不能當成絕對唯一。

程式縮排、詩的換行、字串值的空白可能有意義，不能一律清掉。用統一空白作比較，也不表示訓練原文必須失去空白。副本即使只在訓練側，也會增加抽到該內容的機會，改變資料分配。

練習只把第二段改成 `"鳥 看狗"`。先預測正規化不同內容變3、前兩指紋不相同，最後舊assert會失敗，再執行核對。將檢查改成 `hashes[0] != hashes[1]`，確認一個字變化被保留；如果你改的是多一個空格，則本規則仍視為同一內容。

<details>
<summary>補充：實作約定與原始紀錄</summary>

這個想法已用在[T.4的真實文字實驗](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)：先以合併空白後的完整文字判定重複，對相同內容只留一份。留下的記錄加上完整內容指紋作分組標籤，再切分。TinyStories的512篇仍是512篇，資料包中的365首詩也仍是365首，這一步沒有再刪掉記錄。原文仍保留自己的空白與換行來訓練；合併空白只是比較內容的規則。注意「沒有找到這種重複」只回答了完整內容是否相同，改寫、同一故事的相似版本還要另查，下一節就看這個缺口。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
